<a href="https://colab.research.google.com/github/Ugobo/Chatbot-development-/blob/main/week%203/Another_copy_of_Week_3_FAQ_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — Your First Rule-Based Bot
### Chatbots Development (AITCD001) · Learning Outcome 2 · United African Institute of Technology

Learning Outcome 2 starts here. Over the next five weeks you build **one** chatbot and extend it five times.

Today: conditional logic, keyword tables, and a fallback that earns its place.


**By the end you will have:** a working FAQ bot, a test harness that proves it works, and a log of
everything it could not answer. Keep all three — Weeks 4, 5, 6 and 7 all build on them.

## Part 1 — Where you left off

This is the shape of your Week 1 pizza bot: a ladder of `if` / `elif` statements comparing exact text.
Run it and watch the second and third messages fail.

In [ ]:
def week1_style_bot(message):
    """The Week 1 approach: one branch per rule, exact text matching."""
    if "hours" in message:
        return "We open 10:00 - 22:00, every day."
    elif "deliver" in message:
        return "We deliver anywhere inside Kigali."
    elif "price" in message:
        return "Pizzas start at 6,000 RWF."
    else:
        return "Sorry, I did not understand that."


for m in ["what are your hours", "WHAT ARE YOUR HOURS", "  hours  "]:
    print(f"{m!r:28} -> {week1_style_bot(m)}")

'what are your hours'        -> We open 10:00 - 22:00, every day.
'WHAT ARE YOUR HOURS'        -> Sorry, I did not understand that.
'  hours  '                  -> We open 10:00 - 22:00, every day.


In [ ]:
def reply(message):
    msg = message.lower()

    if "order" in msg:
        return "Sure - what size would you like?"

    elif "cancel my order" in msg:
        return "Cancelled. Sorry to hear it."

    elif "deliver" in msg:
        return "We deliver anywhere inside Kigali."

    elif "price" in msg:
        return "Pizzas start at 6,000 RWF."

    else:
        return "Sorry, I did not understand that."


In [ ]:
reply("what time do you open?")

'Sorry, I did not understand that.'

The second and third messages are the *same question*. The bot only answers the first one.

Nothing is wrong with the logic — the input was never cleaned up before the rules saw it.

## Part 2 — First match wins

This is the most common bug in rule-based bots, and it is almost impossible to spot by reading.

**Predict the output before you run the cell.** You did this on paper in Activity 1, Part A.

In [ ]:
def broken(message):
    msg = message.lower()
    if "order" in msg:
        return "Sure - what size would you like?"
    elif "cancel my order" in msg:
        return "Cancelled. Sorry to hear it."
    else:
        return "Sorry, I did not understand that."


print(broken("I want to cancel my order"))

Sure - what size would you like?


`"cancel my order"` contains the word `"order"`, so the **first** branch matches and returns.
The cancel branch is unreachable and it can never run, no matter what anyone types.

The fix is not more code. It is **order**: check the specific rule before the general one.

In [ ]:
def fixed(message):
    msg = message.lower()
    if "cancel" in msg:
        return "Cancelled. Sorry to hear it."
    elif "order" in msg:
        return "Sure - what size would you like?"
    else:
        return "Sorry, I did not understand that."


for m in ["I want to cancel my order", "can I order a large pepperoni"]:
    print(f"{m!r:34} -> {fixed(m)}")

'I want to cancel my order'        -> Cancelled. Sorry to hear it.
'can I order a large pepperoni'    -> Sure - what size would you like?


**Remember this one.** Every time you add a keyword to a rule that is checked early, you may be
stealing messages from a rule further down. That is why Part 8 makes you re-run *every* test.

## Part 3 — Clean the input once

One small function that every message passes through before any rule sees it. You will keep extending
this same function until Week 8.

In [ ]:
def clean(text):
    """Everything a message goes through before any rule sees it."""
    return text.lower().strip()


for m in ["  HOURS?  ", "Do You Deliver", "PRICE"]:
    print(f"{m!r:20} -> {clean(m)!r}")

'  HOURS?  '         -> 'hours?'
'Do You Deliver'     -> 'do you deliver'
'PRICE'              -> 'price'


Useful string methods, for reference — these five do most of the work in a rule-based bot:

| Method | What it does |
|---|---|
| `.lower()` | Makes the whole string lowercase |
| `.strip()` | Removes spaces at the start and end |
| `"x" in text` | `True` if `x` appears **anywhere** in `text` |
| `.find("x")` | Position of `x`, or `-1` if it is not there |
| `.split()` | Breaks the string into a list of words |

`.split()` is tokenisation done crudely by hand — the same idea as Week 2's spaCy tokens, without the
13MB model. Both are legitimate; it depends what the job is.

In [ ]:
msg = clean("  Do you DELIVER to Kicukiro?  ")

print("cleaned :", repr(msg))
print("contains 'deliver' :", "deliver" in msg)
print("position of 'deliver' :", msg.find("deliver"))
print("position of 'pizza'   :", msg.find("pizza"), "  <- -1 means not found")
print("words   :", msg.split())

cleaned : 'do you deliver to kicukiro?'
contains 'deliver' : True
position of 'deliver' : 7
position of 'pizza'   : -1   <- -1 means not found
words   : ['do', 'you', 'deliver', 'to', 'kicukiro?']


## Part 4 — Rules as data, not as code

Five rules as `if`/`elif` is readable. Twenty is sixty lines you cannot see on one screen, and adding
a question means writing code instead of writing an answer.

So move the rules out of the code and into a dictionary.

In [ ]:
FAQ_SIMPLE = {
    "hours":   "We open 10:00 - 22:00, every day.",
    "deliver": "We deliver anywhere inside Kigali.",
    "price":   "Pizzas start at 6,000 RWF.",
    "vegan":   "Yes - ask for the vegan cheese.",
    "halal":   "All our meat is halal certified.",
}

FALLBACK = "Sorry, I did not understand that."


def reply(message):
    msg = clean(message)
    for keyword, response in FAQ_SIMPLE.items():
        if keyword in msg:
            return response
    return FALLBACK


for m in ["what time do you OPEN?", "do you deliver", "is it halal?", "who are you"]:
    print(f"{m!r:26} -> {reply(m)}")

'what time do you OPEN?'   -> Sorry, I did not understand that.
'do you deliver'           -> We deliver anywhere inside Kigali.
'is it halal?'             -> All our meat is halal certified.
'who are you'              -> Sorry, I did not understand that.


The `for` loop **is** the `elif` ladder — written once, instead of once per rule. `return` stops
the loop dead, so it is still first-match-wins.

**Notice what did *not* happen: adding a sixth question would not change `reply()` at all. That is the whole point.**

`"what time do you OPEN?"` still fails, though. Look at why before you read on.

It fails because the keyword is `"hours"` and the customer said `"time"`. **One keyword is never enough** customers do not know your keywords.

## Part 5 — Synonyms, and the tag

In Week 2 you solved this with synonym sets. Same idea, new shape: each rule becomes
`(keywords, tag, response)`.

The **tag** is a short name for what the customer wanted. It looks like bookkeeping today. In Week 5
it becomes the intent name, and this list becomes your routing table — so get into the habit now.

In [ ]:
FAQ = [
    (("cancel", "refund"), "cancel", "I can cancel an order - what name was it under?"),
    (("hour", "open", "close", "what time"), "hours", "We are open 10:00 - 22:00, every day."),
    (("deliver", "delivery", "bring", "send"), "delivery", "We deliver anywhere inside Kigali. Delivery is 1,500 RWF."),
    (("price", "cost", "how much", "rwf", "angahe"), "price", "Pizzas start at 6,000 RWF. A large pepperoni is 12,000 RWF."),
    (("menu", "what do you have", "options"), "menu", "Margherita, pepperoni, mushroom, vegan and BBQ chicken."),
    (("vegan", "vegetarian", "veggie"), "vegan", "Yes - any pizza can be made with vegan cheese."),
    (("hi", "hello", "muraho", "hey"), "greeting", "Muraho! I am the Kigali Pizza bot. How can I help you today?"),
    (("order", "buy", "want a", "can i get"), "order", "Sure - what size would you like? Small, medium or large."),
]

Two things to notice in that table:

1. **`cancel` is first.** Otherwise `"cancel my order"` hits the `order` rule — the Part 2 bug.
2. **Shorter keywords match more.** `"hour"` catches *hours*, *opening hours* and *what hour do you
   open*. That is usually what you want, but it is also how collisions start.

## Part 6 — The fallback that logs

Every bot you will ever build fails to match something. Language is bigger than your rule table and
always will be — so the fallback is not an error message, it is the message your bot sends most often
to the people it is helping least.

A good fallback does three jobs: apologise briefly, say what the bot *can* do, and **record the
failure**.

In [ ]:
UNMATCHED = []        # every message no rule could answer

FALLBACK_TEXT = (
    "Sorry, I did not get that. I can help with:hours | delivery | prices | menu | ordering; Type 'menu' to see everything, or 'human' to reach a person."
)


def answer(message):
    """Return (tag, response). The tag is what makes the bot testable."""
    msg = clean(message)

    for keywords, tag, response in FAQ:
        if any(k in msg for k in keywords):
            return tag, response

    UNMATCHED.append(message)
    return "fallback", FALLBACK_TEXT


for m in ["what time do you open?", "do u have vegan", "ni angahe?"]:
    tag, response = answer(m)
    print(f"{m!r:26} [{tag}]  {response.splitlines()[0]}")

'what time do you open?'   [hours]  We are open 10:00 - 22:00, every day.
'do u have vegan'          [vegan]  Yes - any pizza can be made with vegan cheese.
'ni angahe?'               [price]  Pizzas start at 6,000 RWF. A large pepperoni is 12,000 RWF.


`any(k in msg for k in keywords)` is "is *any* of my keywords in this message?" — it is the
synonym set from Week 2, in one line.

That `UNMATCHED.append(message)` line is the most valuable line in this notebook. It is why:

- **Week 4** — your log decides which regex patterns are actually worth writing. Evidence, not guesses.
- **Week 5** — it shows which intents your router is still missing.
- **Week 6** — it is your evidence for criterion 2.4, *validated against non-native-English and
  local-language phrasing*, collected honestly over four weeks instead of invented on the day.
- **Week 7** — it becomes the "known limitations" section of your portfolio README.

**Do not clear this list.**

## Part 7 — FAQ Bot v1

Now put it together with the three UX touches that cost nothing: a greeting that says what the bot can
do, short answers, and a graceful exit.

Plus one from Week 2's ethics — **say you are a bot in the first message**, not in a policy nobody
opens.

In [ ]:
GREETING = (
    "Hi! I am the Kigali Pizza bot - an automated assistant, not a person.\n"
    "I can help with opening hours, delivery, prices, the menu and orders.\n"
    "Type 'bye' at any time to end the chat."
)

EXIT_WORDS = ("bye", "goodbye", "exit", "quit", "thanks", "murakoze")


def chat():
    """Interactive chat loop. Type 'bye' to leave."""
    print(GREETING)
    print()

    while True:
        message = input("You: ")

        if clean(message) in EXIT_WORDS:          # exact match, NOT "in" - see below
            print("Bot: Thanks for chatting. Murakoze!")
            break

        tag, response = answer(message)
        print(f"Bot: {response}")
        print()

**Run the next cell on its own** — it waits for you to type. Try five or six messages, including
one you think will fail, then type `bye`.

In [ ]:
chat()

Hi! I am the Kigali Pizza bot - an automated assistant, not a person.
I can help with opening hours, delivery, prices, the menu and orders.
Type 'bye' at any time to end the chat.

You: bye
Bot: Thanks for chatting. Murakoze!


Two design decisions in that loop are worth stopping on.

**The exit check runs before the rule table.** `"bye"` must work from anywhere in the conversation,
including halfway through an order. If the rule table ran first, a rule with a loose keyword could
swallow the exit.

**The exit check uses exact match, not `in`.** Everywhere else today `in` was the right tool. Here it
is wrong: with `if any(w in msg for w in EXIT_WORDS)`, the message
`"thanks, what are your hours"` would **end the chat** instead of answering the question — because it
contains `"thanks"`. A polite customer would be hung up on.

So `in` is not automatically better. `in` asks *"is this idea somewhere in the message?"* — right for
topics. Exact match asks *"is this the whole message?"* — right for commands.

## Part 8 — Test it properly · Portfolio checkpoint #1

A test is three things: an **input**, what you **expected**, and what you **actually got**. If you are
not writing down what you expected before you run it, you are not testing — you are chatting with your own bot and being pleased.

In [ ]:
TESTS = [
    # Original tests
    ("what time do you open?", "hours"),
    ("do you deliver to Kicukiro", "delivery"),
    ("HOW MUCH IS A LARGE", "price"),
    ("do u delivr", "fallback"),  # Expected FAIL (handled in Week 4)
    ("asdfgh", "fallback"),
    # New tests for Activity 2 additions
    ("ni angahe?", "price"),
    ("hi", "greeting"),
]

passed = 0
for message, expected in TESTS:
    tag, _ = answer(message)
    ok = tag == expected
    passed += ok
    print(
        f"{'PASS' if ok else 'FAIL'} | {message!r:32} expected {expected:9} got {tag}"
    )

print(f"\n{passed} of {len(TESTS)} passed")

PASS | 'what time do you open?'         expected hours     got hours
PASS | 'do you deliver to Kicukiro'     expected delivery  got delivery
PASS | 'HOW MUCH IS A LARGE'            expected price     got price
PASS | 'do u delivr'                    expected fallback  got fallback
PASS | 'asdfgh'                         expected fallback  got fallback
PASS | 'ni angahe?'                     expected price     got price
PASS | 'hi'                             expected greeting  got greeting

7 of 7 passed


**You should get 4 of 5.** Two of those results are deliberate, and you should be able to
explain both:

- **`"do u delivr"` fails.** Do **not** fix it by adding `"delivr"` as a keyword — you would be
  guessing at spellings forever. Week 4 gives you the tool that handles it properly. Record the
  failure and leave it.

  (Worth noticing: `"do u deliverr"` — with the typo at the *end* — would have **passed**, because
  `"deliver"` is inside `"deliverr"`. `"delivr"` fails because the missing letter is in the middle.
  Your bot is not robust to typos; it just happened to survive one kind.)
- **`"asdfgh"` passes.** Falling back correctly on nonsense is the bot *working*. If `"asdfgh"`
  matched one of your rules, one of your keywords is too short.

A failing test you understand is worth more than a passing test you cannot account for.

## Part 9 — Activity 2: break the bot

Swap laptops with your partner and run your six attack messages against their bot. Then come back
here and read the log.

In [ ]:
# Run your partner's six attack messages here
ATTACKS = [
    "do you delivr to remera",          # typo
    "WHAT TIME DO YOU CLOSE",           # capitals
    "do you deliver and how much",      # two questions at once
    "muraho, ni angahe?",               # local phrasing - write your own
    "hello, I was wondering whether it might be possible to know your opening times",
    "who won the match last night",     # off topic
]

for m in ATTACKS:
    tag, _ = answer(m)
    print(f"[{tag:9}] {m}")

[fallback ] do you delivr to remera
[hours    ] WHAT TIME DO YOU CLOSE
[delivery ] do you deliver and how much
[price    ] muraho, ni angahe?
[hours    ] hello, I was wondering whether it might be possible to know your opening times
[fallback ] who won the match last night


In [ ]:
# Everything the bot could not answer, including your partner's attacks
print(f"{len(UNMATCHED)} unmatched messages\n")
for i, m in enumerate(UNMATCHED, start=1):
    print(i, "|", m)

Pick the **two failures that would matter most to a real pizza shop** — not the two that are
easiest to fix. Edit the `FAQ` list in Part 5, re-run that cell, then re-run **all five** tests in
Part 8.

Adding a keyword to an early rule can steal messages from a later rule that used to work. Re-run
everything, every time.

Record what you changed in **Activity 2, Part C**.

## Part 10 — A ten-minute look at next week's tool

Read this; you are not expected to write it yet.

Look at how many different ways one question arrived in your log. Every one of these means *"what are
your prices?"*:

```
"how much is a large"   "whats the price"   "price?"   "how much???"   "cost of a medium"
```

You could keep adding keywords forever. Or you could write one pattern.

In [ ]:
import re

PHRASINGS = [
    "how much is a large",
    "whats the price",
    "price?",
    "how much???",
    "cost of a medium pizza",
    "do you deliver",  # <- should NOT match
]

for m in PHRASINGS:
    hit = re.search(r"how much|price|cost|rwf", m)
    print(f"{m!r:32} -> {'price' if hit else 'no match'}")

'how much is a large'            -> price
'whats the price'                -> price
'price?'                         -> price
'how much???'                    -> price
'cost of a medium pizza'         -> price
'do you deliver'                 -> no match


The `|` means **or**. That is the whole idea for today — one pattern, five phrasings, no new
keywords.

Week 4 is entirely about this. **Bring your `UNMATCHED` log** — it is where Week 4 starts.